In [1]:
import json, glob
from pathlib import Path
import pandas as pd

records = []
for f in sorted(Path("../data/async").glob("*/*/*.json")):
    if f.name == "manifest.json":
        continue
    rec = json.load(open(f))
    rec["model_dir"] = f.parts[-3]   # e.g. gpt-4o-mini
    rec["energy_mode"] = f.parts[-2] # objective_energy_seq / subjective_energy_seq
    rec["file"] = f.name
    records.append(rec)

df = pd.json_normalize(records)

parts = df["file"].str.extract(r"(?P<question>.+)__(?P<graph>[^_]+)__rep(?P<rep>\d+)\.json")
df = df.join(parts)
df["replica"] = df["rep"].astype(int)

df["model"] = df["meta.model"]

# load them and add to the df here
obj = pd.concat([
    pd.read_json("../data/obj/train.jsonl", lines=True),
    pd.read_json("../data/obj/test.jsonl", lines=True),
])
ground_truth = obj.set_index("qid")["answer"]              # "A" or "B"
lean = json.load(open("../data/subj/lean.json"))["lean"]   # qid -> left/right/ambiguous

df["ground_truth"] = df["question"].map(ground_truth)      # NaN on subjective rows
df["political_lean"] = df["question"].map(lean)            # NaN on objective rows

# sanity check: every row got exactly one of the two labels
assert (df["ground_truth"].notna() ^ df["political_lean"].notna()).all()
cols = ['model', 'mode', 'statement', 'J', 'spins_history', 'spins_raw_history',
        'replica', 'ground_truth', 'political_lean']
df[cols].to_json("../data/clean_runs_seq.json", orient="records")

from datasets import Dataset
ds = Dataset.from_pandas(df[cols].reset_index(drop=True))
ds.push_to_hub("physics-of-agents/agent-opinions-async", private=False)

/Users/batuel/Documents/Courses/ee269/ee269_project/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 24.30ba/s]
Processing Files (1 / 1): 100%|██████████| 10.9MB / 10.9MB, 1.02MB/s  
New Data Upload: 100%|██████████| 9.02MB / 9.02MB,  844kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.44s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/physics-of-agents/agent-opinions-async/commit/19464407cce5381847eb8181d721abac1e629f5d', commit_message='Upload dataset', commit_description='', oid='19464407cce5381847eb8181d721abac1e629f5d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/physics-of-agents/agent-opinions-async', endpoint='https://huggingface.co', repo_type='dataset', repo_id='physics-of-agents/agent-opinions-async'), pr_revision=None, pr_num=None)